In [1]:
import sys
sys.path.append("../src")

from OmiScraper import OmiScraper
from AreasDictionaries import university_areas
from AreasDictionaries import area_university_list
from Functions import area_uni_finder, area_check

import pandas as pd
from io import StringIO

## 1. Manually creating the area-level datasets.
The original idea for the study was to analyze only a few areas, specifically those I was already familiar with in Rome, where I had previously owned and managed real estate.  
At this stage, manually entering the html code every time and building a small number of datasets seemed still achievable.

Each area-level dataset follows a standardized structure, combining the original OMI table with additional contextual features:

Original OMI data (from HTML table):

- Province -> Rome
- Area -> Name of the area (e.g. San Paolo)
- Tipologia
- Stato conservativo
- Valori Compravendita (Min / Max)
- Valori Locazione (Min / Max)
- Superficie

Additional engineered features (from OMI dataset):
From the OMI website:
- Municipality -> Rome
- Fascia/Zona -> OMI zone description
- Codice Zona -> OMI zone code
- Microzona Catastale -> cadastral microzone
- Tipologia Prevalente -> predominant property type
- Destinazione -> usage (residential)

Additional engineered features (from indipendent reaserch):
- Area -> Name of the area (e.g. San Paolo)
- Universities -> nearby universities
- Proximity -> relative proximity to universities (e.g. core, nearby)
- Type -> classification of universities (e.g. public, specialized)

These additional fields were manually assigned to enrich the raw OMI data and enable further analysis across areas.

In [9]:
#prices_sanpaolo

html_table1 = """ <table class="table table-striped table-hover table-header table-bordered" summary="La tabella riporta i valori delle quotazioni del mercato immobiliare per il semestre selezionato">
<thead><tr><th rowspan="2">Tipologia</th><th rowspan="2">Stato conservativo</th><th id="vm" colspan="2">Valori Compravendita (€/mq)</th><th rowspan="2">Superficie (L/N)</th><th id="vl" colspan="2">Valori Locazione (€/mq x mese)</th><th rowspan="2">Superficie (L/N)</th></tr><tr><th id="vmmin">Min</th><th id="vmmax">Max</th><th id="vlmin">Min</th><th id="vlmax">Max</th></tr></thead><tbody><tr><td class="sin">Abitazioni civili</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">2800</td><td class="dx" headers="vm vmmax">4000</td><td class="center">L</td><td class="dx" headers="vl vlmax">10,3</td><td class="dx" headers="vl vlmax">14,8</td><td class="center">L</td></tr></tbody><tbody><tr><td class="sin">Abitazioni di tipo economico</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">2550</td><td class="dx" headers="vm vmmax">3500</td><td class="center">L</td><td class="dx" headers="vl vlmax">9</td><td class="dx" headers="vl vlmax">12,8</td><td class="center">L</td></tr></tbody><tbody><tr><td class="sin">Box</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">1300</td><td class="dx" headers="vm vmmax">1850</td><td class="center">L</td><td class="dx" headers="vl vlmax">6,3</td><td class="dx" headers="vl vlmax">9,3</td><td class="center">L</td></tr></tbody><tbody><tr><td class="sin">Posti auto coperti</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">1000</td><td class="dx" headers="vm vmmax">1400</td><td class="center">L</td><td class="dx" headers="vl vlmax">5,3</td><td class="dx" headers="vl vlmax">7,5</td><td class="center">L</td></tr></tbody><tbody><tr><td class="sin">Posti auto scoperti</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">550</td><td class="dx" headers="vm vmmax">800</td><td class="center">L</td><td class="dx" headers="vl vlmax">3,3</td><td class="dx" headers="vl vlmax">4,8</td><td class="center">L</td></tr></tbody></table> """

prices_sanpaolo = pd.read_html(html_table1)[0]

prices_sanpaolo['Area'] = 'San Paolo'
prices_sanpaolo['Province'] = pd.Series(['Rome' for i in range(len(prices_sanpaolo))])
prices_sanpaolo['Municipality'] = 'Rome'
prices_sanpaolo['Fascia/Zona'] = ['Periferica/San Paolo (Tullio Levi Civita)' for i, x in enumerate(prices_sanpaolo.index)]
prices_sanpaolo['Codice Zona'] = 'D5'
prices_sanpaolo['Microzona Catastale'] = 53
prices_sanpaolo['Tipologia Prevalente'] = 'Abitazioni di tipo economico'
prices_sanpaolo['Destinazione'] = 'Residenziale'
prices_sanpaolo['Universities'] = 'RM3'
prices_sanpaolo['Proximity'] = 'Core'
prices_sanpaolo['Type'] = 'public'

/var/folders/gs/d24257s15q1bzbn_vyvcxhb00000gp/T/ipykernel_80666/793719241.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  prices_sanpaolo = pd.read_html(html_table)[0]


In [11]:
#prices_garbatella

html_table2 = """<table class="table table-striped table-hover table-header table-bordered" summary="La tabella riporta i valori delle quotazioni del mercato immobiliare per il semestre selezionato">
<thead><tr><th rowspan="2">Tipologia</th><th rowspan="2">Stato conservativo</th><th id="vm" colspan="2">Valori Compravendita (€/mq)</th><th rowspan="2">Superficie (L/N)</th><th id="vl" colspan="2">Valori Locazione (€/mq x mese)</th><th rowspan="2">Superficie (L/N)</th></tr><tr><th id="vmmin">Min</th><th id="vmmax">Max</th><th id="vlmin">Min</th><th id="vlmax">Max</th></tr></thead><tbody><tr><td class="sin">Abitazioni civili</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">2800</td><td class="dx" headers="vm vmmax">3900</td><td class="center">L</td><td class="dx" headers="vl vlmax">11,8</td><td class="dx" headers="vl vlmax">16</td><td class="center">L</td></tr></tbody><tbody><tr><td class="sin">Abitazioni di tipo economico</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">2450</td><td class="dx" headers="vm vmmax">3300</td><td class="center">L</td><td class="dx" headers="vl vlmax">10,3</td><td class="dx" headers="vl vlmax">14,3</td><td class="center">L</td></tr></tbody><tbody><tr><td class="sin">Box</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">1500</td><td class="dx" headers="vm vmmax">2100</td><td class="center">L</td><td class="dx" headers="vl vlmax">8</td><td class="dx" headers="vl vlmax">11,5</td><td class="center">L</td></tr></tbody><tbody><tr><td class="sin">Posti auto coperti</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">1100</td><td class="dx" headers="vm vmmax">1600</td><td class="center">L</td><td class="dx" headers="vl vlmax">6,5</td><td class="dx" headers="vl vlmax">9</td><td class="center">L</td></tr></tbody><tbody><tr><td class="sin">Posti auto scoperti</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">650</td><td class="dx" headers="vm vmmax">900</td><td class="center">L</td><td class="dx" headers="vl vlmax">4</td><td class="dx" headers="vl vlmax">5,8</td><td class="center">L</td></tr></tbody></table>"""
prices_garbatella = pd.read_html(html_table2)[0]

prices_garbatella = prices_garbatella.assign(**
    {
        'Area': 'Garbatella',
        'Province': 'Rome',
        'Municipality': 'Rome',
        'Fascia/Zona': 'Semicentrale/GARBATELLA (LARGO DELLE SETTE CHIESE)',
        'Codice Zona': 'C10',
        'Microzona Catastale': 52,
        'Tipologia prevalente': 'Abitazioni civili',
        'Destinazione': 'Residenziale',
        'Universities': 'RM3',
        'Proximity': 'core',
        'Type': 'public'
        
    }
)

/var/folders/gs/d24257s15q1bzbn_vyvcxhb00000gp/T/ipykernel_80666/2594404970.py:5: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  prices_garbatella = pd.read_html(html_table3)[0]


## 2. Creation of a supporting dataset: Areas vs. Universities
As the project evolved, I expanded the number of areas included in the analysis.

To support this, I created a dictionary mapping each area to its related universities, along with proximity. This auxiliary dataset serves as a reference for the creation of the main dataset with location-based features.

***Note:*** For presentation purposes, only the first five records are shown below.

In [42]:
area_uni = pd.DataFrame(area_university_list, columns = ['Area', 'University','Proximity', 'Type'])
area_uni.head(5)

,Area,University,Proximity,Type
0,San Lorenzo,Sapienza,core,public
1,Policlinico,Sapienza,core,public
2,Piazza Bologna,Sapienza,core,public
3,Nomentano,Sapienza,core,public
4,Verano,Sapienza,core,public


### 2.1. Number of universities in proximity to each area
This metric serves as a quick reference for the next steps to better identify the number of universities associated with each area based on the previously defined mapping.

***Note:*** For presentation purposes, only the first ten records are shown.

In [54]:
area_uni['Area'].value_counts().head(10)

Area
Monteverde          4
Testaccio           3
Trastevere          3
Parioli             2
Ostiense            2
Re di Roma          2
Pigneto             2
Gianicolo           2
Nomentano           2
Pineta Sacchetti    1
Name: count, dtype: int64

## 3. Designing a scalable data pipeline from the area-level datasets
As previously mentioned, I decided to expand the number of areas included in the analysis. At that point, the initial manual approach became inefficient and difficult to scale.  

To address this, I developed a class, **OmiScraper**, to standardize and automate the insertion and processing of the HTML tables. The class is defined in the project’s source code (**src/**) and imported into this notebook. This approach enables a faster, more consistent and reusable construction of a larger dataset as the number of areas increases. 

***Note:*** As mentioned in the README section, the OMI website does not support reliable web scraping, so the process still requires manual retrieval of the HTML tables.

### 3.1 Using pre-implemented area-level datasets

I structured the class so that it could take the datasets that were already manually created, without losing the work done up to this point (***Note***).

This is done by initializing the class with a dictionary (**initial_data**) containing the existing area-level datasets, which are then integrated into the main structure.

***Note:*** Initially, at least eight areas were created manually. As these followed the same approach, only two are included in the final version of the project to illustrate the workflow.

In [13]:
initial_data = {
    "San Paolo": prices_sanpaolo,
    "Garbatella": prices_garbatella,
}

In [15]:
pd.concat(initial_data, ignore_index = True)

Tipologia Stato conservativo  \
                      Tipologia Stato conservativo   
0             Abitazioni civili            NORMALE   
1  Abitazioni di tipo economico            NORMALE   
2                           Box            NORMALE   
3            Posti auto coperti            NORMALE   
4           Posti auto scoperti            NORMALE   
5             Abitazioni civili            NORMALE   
6  Abitazioni di tipo economico            NORMALE   
7                           Box            NORMALE   
8            Posti auto coperti            NORMALE   
9           Posti auto scoperti            NORMALE   

  Valori Compravendita (€/mq)       Superficie (L/N)  \
                          Min   Max Superficie (L/N)   
0                        2800  4000                L   
1                        2550  3500                L   
2                        1300  1850                L   
3                        1000  1400                L   
4                         550   800                L   
5                        2800  3900                L   
6                        2450  3300                L   
7                        1500  2100                L   
8                        1100  1600                L   
9                         650   900                L   

  Valori Locazione (€/mq x mese)        Superficie (L/N)        Area Province  \
                             Min  Max Superficie (L/N).1                        
0                            103  148                  L   San Paolo     Rome   
1                              9  128                  L   San Paolo     Rome   
2                             63   93                  L   San Paolo     Rome   
3                             53   75                  L   San Paolo     Rome   
4                             33   48                  L   San Paolo     Rome   
5                            118   16                  L  Garbatella     Rome   
6                            103  143                  L  Garbatella     Rome   
7                              8  115                  L  Garbatella     Rome   
8                             65    9                  L  Garbatella     Rome   
9                              4   58                  L  Garbatella     Rome   

  Municipality                                        Fascia/Zona Codice Zona  \
                                                                                
0         Rome          Periferica/San Paolo (Tullio Levi Civita)          D5   
1         Rome          Periferica/San Paolo (Tullio Levi Civita)          D5   
2         Rome          Periferica/San Paolo (Tullio Levi Civita)          D5   
3         Rome          Periferica/San Paolo (Tullio Levi Civita)          D5   
4         Rome          Periferica/San Paolo (Tullio Levi Civita)          D5   
5         Rome  Semicentrale/GARBATELLA (LARGO DELLE SETTE CHI...         C10   
6         Rome  Semicentrale/GARBATELLA (LARGO DELLE SETTE CHI...         C10   
7         Rome  Semicentrale/GARBATELLA (LARGO DELLE SETTE CHI...         C10   
8         Rome  Semicentrale/GARBATELLA (LARGO DELLE SETTE CHI...         C10   
9         Rome  Semicentrale/GARBATELLA (LARGO DELLE SETTE CHI...         C10   

  Microzona Catastale          Tipologia Prevalente  Destinazione  \
                                                                    
0                  53  Abitazioni di tipo economico  Residenziale   
1                  53  Abitazioni di tipo economico  Residenziale   
2                  53  Abitazioni di tipo economico  Residenziale   
3                  53  Abitazioni di tipo economico  Residenziale   
4                  53  Abitazioni di tipo economico  Residenziale   
5                  52                           NaN  Residenziale   
6                  52                           NaN  Residenziale   
7                  52                           NaN  Residenziale   
8                  52                           NaN  Residenzial

### 3.2 Handling MultiIndex columns
The OMI tables contain hierarchical column headers (MultiIndex), which are not suitable for concatenation and further analysis.

To address this, I implemented a method to flatten the column structure into a single level, ensuring consistency across all datasets and enabling proper merging.

In [19]:
dt = OmiScraper(initial_data)
dt.flatten_all()

In [29]:
dt.current_dt()

,Tipologia,Stato conservativo,Valori Compravendita (€/mq) - Min,Valori Compravendita (€/mq) - Max,Superficie (L/N),Valori Locazione (€/mq x mese) - Min,Valori Locazione (€/mq x mese) - Max,Superficie (L/N) - Superficie (L/N).1,Area,Province,Municipality,Fascia/Zona,Codice Zona,Microzona Catastale,Tipologia Prevalente,Destinazione,Universities,Proximity,Type,Tipologia prevalente
0,Abitazioni civili,NORMALE,2800,4000,L,103,148,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
1,Abitazioni di tipo economico,NORMALE,2550,3500,L,9,128,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
2,Box,NORMALE,1300,1850,L,63,93,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
3,Posti auto coperti,NORMALE,1000,1400,L,53,75,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
4,Posti auto scoperti,NORMALE,550,800,L,33,48,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
5,Abitazioni civili,NORMALE,2800,3900,L,118,16,L,Garbatella,Rome,Rome,Semicentrale/GARBATELLA (LARGO DELLE SETTE CHI...,C10,52,NaN,Residenziale,RM3,core,public,Abitazioni civili
6,Abitazioni di tipo economico,NORMALE,2450,3300,L,103,143,L,Garbatella,Rome,Rome,Semicentrale/GARBATELLA (LARGO DELLE SETTE CHI...,C10,52,NaN,Residenziale,RM3,core,public,Abitazioni civili
7,Box,NORMALE,1500,2100,L,8,115,L,Garbatella,Rome,Rome,Semicentrale/GARBATELLA (LARGO DELLE SETTE CHI...,C10,52,NaN,Residenziale,RM3,core,public,Abitazioni civili
8,Posti auto coperti,NORMALE,1100,1600,L,65,9,L,Garbatella,Rome,Rome,Semicentrale/GARBATELLA (LARGO DELLE SETTE CHI...,C10,52,NaN,Residenziale,RM3,core,public,Abitazioni civili
9,Posti auto scoperti,NORMALE,650,900,L,4,58,L,Garbatella,Rome,Rome,Semicentrale/GARBATELLA (LARGO DELLE SETTE CHI...,C10,52,NaN,Residenziale,RM3,core,public,Abitazioni civili


### 3.3 Feature extraction: universities by area
Using the dataset created earlier, I implemented a simple function, **area_uni_finder()**, to retrieve the universities (and their features) associated with each area.

This allows to have a reference and to faster insert all the necessary features for each area

In [637]:
area_uni_finder()

Select the desired area from the dataset:  testaccio


,Area,University,Proximity,Type
16,Testaccio,Roma Tre,nearby,public
47,Testaccio,John Cabot,nearby,international
52,Testaccio,IED,core,specialized


## 4. Dataset construction workflow
In this phase, I used the **.add_dt_col()** method implemented in the class to add each OMI table to the dataset, along with additional metadata that is available on the page but not included directly in the HTML table.

Although this step still required manual retrieval of the HTML code, the class significantly streamlined the process, allowing for a consistent and structured insertion of each area-level dataset while using the previously defined areas–universities mapping as a reference.

The method requires the following inputs:

- area_name -> name of the area
- html_table -> HTML code of the OMI table
- zona -> OMI zone description
- codice_zona -> OMI zone code
- microzona_catastale -> cadastral microzone
- tipologia_prevalente -> predominant property type
- universities -> associated universities
- proximity -> relative proximity to universities
- type -> classification of universities

In [653]:
html_code = """<table class="table table-striped table-hover table-header table-bordered" summary="La tabella riporta i valori delle quotazioni del mercato immobiliare per il semestre selezionato">
<thead><tr><th rowspan="2">Tipologia</th><th rowspan="2">Stato conservativo</th><th id="vm" colspan="2">Valori Compravendita (€/mq)</th><th rowspan="2">Superficie (L/N)</th><th id="vl" colspan="2">Valori Locazione (€/mq x mese)</th><th rowspan="2">Superficie (L/N)</th></tr><tr><th id="vmmin">Min</th><th id="vmmax">Max</th><th id="vlmin">Min</th><th id="vlmax">Max</th></tr></thead><tbody><tr><td class="sin">Abitazioni civili</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">2450</td><td class="dx" headers="vm vmmax">3400</td><td class="center">L</td><td class="dx" headers="vl vlmax">10,5</td><td class="dx" headers="vl vlmax">15</td><td class="center">L</td></tr></tbody><tbody><tr><td class="sin">Box</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">1350</td><td class="dx" headers="vm vmmax">1900</td><td class="center">L</td><td class="dx" headers="vl vlmax">6</td><td class="dx" headers="vl vlmax">8,5</td><td class="center">L</td></tr></tbody><tbody><tr><td class="sin">Posti auto coperti</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">1000</td><td class="dx" headers="vm vmmax">1450</td><td class="center">L</td><td class="dx" headers="vl vlmax">5,3</td><td class="dx" headers="vl vlmax">7,5</td><td class="center">L</td></tr></tbody><tbody><tr><td class="sin">Posti auto scoperti</td><td class="sin">NORMALE</td><td class="dx" headers="vm vmmin">600</td><td class="dx" headers="vm vmmax">900</td><td class="center">L</td><td class="dx" headers="vl vlmax">3,3</td><td class="dx" headers="vl vlmax">4,8</td><td class="center">L</td></tr></tbody></table>"""

In [655]:
dt.add_dt_col(area_name = 'Tintoretto', 
              html_code,
              zona = 'Periferica/TINTORETTO (VIA BALLARIN)',
              codice_zona = 'D38',
              microzona_catastale = 122,
              tipologia_prevalente = 'Abitazioni Civili',
              universities = 'RM3, TVU',
              proximity = 'nearby, extended',
              type1 = 'public, public'              
             )

,Tipologia,Stato conservativo,Valori Compravendita (€/mq) - Min,Valori Compravendita (€/mq) - Max,Superficie (L/N),Valori Locazione (€/mq x mese) - Min,Valori Locazione (€/mq x mese) - Max,Superficie (L/N) - Superficie (L/N).1,Area,Province,Municipality,Fascia/Zona,Codice Zona,Microzona Catastale,Tipologia prevalente,Destinazione,Universities,Proximity,Type
0,Abitazioni civili,NORMALE,2450,3400,L,105,15,L,Tintoretto,Rome,Rome,Periferica/TINTORETTO (VIA BALLARIN),D38,122,Abitazioni Civili,Residenziale,"RM3, TVU","nearby, extended","public, public"
1,Box,NORMALE,1350,1900,L,6,85,L,Tintoretto,Rome,Rome,Periferica/TINTORETTO (VIA BALLARIN),D38,122,Abitazioni Civili,Residenziale,"RM3, TVU","nearby, extended","public, public"
2,Posti auto coperti,NORMALE,1000,1450,L,53,75,L,Tintoretto,Rome,Rome,Periferica/TINTORETTO (VIA BALLARIN),D38,122,Abitazioni Civili,Residenziale,"RM3, TVU","nearby, extended","public, public"
3,Posti auto scoperti,NORMALE,600,900,L,33,48,L,Tintoretto,Rome,Rome,Periferica/TINTORETTO (VIA BALLARIN),D38,122,Abitazioni Civili,Residenziale,"RM3, TVU","nearby, extended","public, public"


### 4.1. Listing available areas
To facilitate the insertion of new datasets, I implemented a method that returns the list of areas currently included in the dataset.

This allows for a quick check before adding a new area, helping avoid duplicates and ensuring consistency during the data construction process.

In [641]:
dt.show_dt_list()

The list of the available areas is: 



['San Paolo',
 'Testaccio',
 'Ostiense',
 'Garbatella',
 'Marconi',
 'San Lorenzo',
 'Monteverde Nuovo',
 'Monteverde Vecchio',
 'Parioli',
 'Trastevere',
 'Pigneto',
 'Nomentano',
 'Re di Roma',
 'Pineta Sacchetti',
 'Monte Mario',
 'Trionfale',
 'Pinciano',
 'Salario',
 'Balduina',
 'Aurelio - Monte di Creta',
 'EUR',
 'Laurentina - Fonte Ostiense',
 'Flaminio - G. Reni',
 'Flaminio - Piazza del Popolo',
 'C. Storico - Tridente',
 'C. Storico - C. Vittorio',
 'Bologna',
 'Trieste',
 'Anagnina - Valle Marciana',
 'Appio Claudio',
 'Don Bosco',
 'Cinecitta',
 'Romanina',
 'Casal Bertone',
 'Furio Camillo - Appio Latino',
 'Monti Tiburtini - Pietralata',
 'Portuense',
 'Prati',
 'Tor Vergata',
 'Tor Pignattara',
 'Colli Aniene',
 'Prenestina',
 'Villaggio Olimpico',
 'Tor Marancia',
 'Montagnola',
 'Pietralata',
 'Roma 70',
 'Quartiere Africano',
 'Porta Portese']

### 4.2. Data validation: handling duplicate area insertions
To avoid inserting the same area multiple times, I implemented a simple validation check within the class to flag areas that had already been added to the dataset.

***Note:*** Due to inconsistencies between the area names used in the supporting dataset and those used in the final dataset, this method was only partially effective. While it was useful in the early stages, when naming conventions were more aligned, it became less reliable as the dataset expanded. It is therefore used mainly for illustrative purposes.

In [583]:
area_uni['Area'].tolist()

['San Lorenzo',
 'Policlinico',
 'Piazza Bologna',
 'Nomentano',
 'Verano',
 'Tiburtina (near station)',
 'Casal Bertone',
 'Portonaccio',
 'Pigneto',
 'Re di Roma',
 'Monti Tiburtini',
 'Furio Camillo',
 'Ostiense',
 'Garbatella',
 'San Paolo',
 'Marconi',
 'Testaccio',
 'Piramide',
 'Portuense',
 'Monteverde',
 'EUR (north edge)',
 'Tor Vergata',
 'Romanina',
 'Anagnina',
 'Cinecittà',
 'Don Bosco',
 'Appio Claudio',
 'Parioli',
 'Trieste',
 'Salario',
 'Nomentano',
 'Pinciano',
 'Monte Mario',
 'Gemelli',
 'Trionfale',
 'Balduina',
 'Pineta Sacchetti',
 'EUR',
 'Laurentina',
 'Aurelia',
 'Bravetta',
 'Monteverde',
 'Monteverde',
 'Gianicolense',
 'Trastevere',
 'Trastevere',
 'Gianicolo',
 'Testaccio',
 'Trastevere',
 'Gianicolo',
 'Monteverde',
 'San Giovanni',
 'Testaccio',
 'Ostiense',
 'Pigneto',
 'Re di Roma',
 'Flaminio',
 'Parioli',
 'Centro Storico',
 'Prati']

In [485]:
areas_loaded = dt.show_dt_list()
area_uni_remaining = area_uni[~area_uni["Area"].isin(dt.show_dt_list())]
area_uni_remaining["Area"]


The list of the available areas is: 

The list of the available areas is: 



1                  Policlinico
2               Piazza Bologna
4                       Verano
5     Tiburtina (near station)
7                  Portonaccio
10             Monti Tiburtini
11               Furio Camillo
17                    Piramide
19                  Monteverde
20            EUR (north edge)
23                    Anagnina
24                   Cinecittà
33                     Gemelli
38                  Laurentina
39                     Aurelia
40                    Bravetta
41                  Monteverde
42                  Monteverde
43                Gianicolense
46                   Gianicolo
49                   Gianicolo
50                  Monteverde
51                San Giovanni
56                    Flaminio
58              Centro Storico
Name: Area, dtype: object

## 5. Final dataset and export to CSV

In [659]:
final_dt = dt.current_dt()
final_dt

,Tipologia,Stato conservativo,Valori Compravendita (€/mq) - Min,Valori Compravendita (€/mq) - Max,Superficie (L/N),Valori Locazione (€/mq x mese) - Min,Valori Locazione (€/mq x mese) - Max,Superficie (L/N) - Superficie (L/N).1,Area,Province,Municipality,Fascia/Zona,Codice Zona,Microzona Catastale,Tipologia Prevalente,Destinazione,Universities,Proximity,Type,Tipologia prevalente
0,Abitazioni civili,NORMALE,2800,4000,L,103,148,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
1,Abitazioni di tipo economico,NORMALE,2550,3500,L,9,128,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
2,Box,NORMALE,1300,1850,L,63,93,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
3,Posti auto coperti,NORMALE,1000,1400,L,53,75,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
4,Posti auto scoperti,NORMALE,550,800,L,33,48,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,Posti auto scoperti,NORMALE,600,900,L,33,48,L,Centocelle,Rome,Rome,Periferica/CENTOCELLE (PIAZZA DEI MIRTI),D14,111,NaN,Residenziale,SAP,extended,public,Abitazioni di Tipo Economico
236,Abitazioni civili,NORMALE,2450,3400,L,105,15,L,Tintoretto,Rome,Rome,Periferica/TINTORETTO (VIA BALLARIN),D38,122,NaN,Residenziale,"RM3, TVU","nearby, extended","public, public",Abitazioni Civili
237,Box,NORMALE,1350,1900,L,6,85,L,Tintoretto,Rome,Rome,Periferica/TINTORETTO (VIA BALLARIN),D38,122,NaN,Residenziale,"RM3, TVU","nearby, extended","public, public",Abitazioni Civili
238,Posti auto coperti,NORMALE,1000,1450,L,53,75,L,Tintoretto,Rome,Rome,Periferica/TINTORETTO (VIA BALLARIN),D38,122,NaN,Residenziale,"RM3, TVU","nearby, extended","public, public",Abitazioni Civili


In [ ]:
final_dt.to_csv("investement_df.csv", index=False)
# final_dt.to_pickle("investement_df.pkl")

---
****This dataset serves as the foundation for the subsequent real estate analysis.****